# 02 - Eksperimen Klasik (Skenario 1-9)

SVM & Random Forest dengan feature engineering manual. Tiap skenario
mengubah **satu variabel** untuk isolasi efek (restorasi, enhancement,
segmentasi, fitur, classifier). Jalankan **berurutan** setelah `01-eda.ipynb`.

In [1]:
# ============================================================
# Setup cell - Kaggle Notebooks (Kaggle-only). Jalankan PALING ATAS.
# Cara attach dataset: panel kanan > + Add Data > cari
#   'fruit and vegetable disease healthy vs rotten' > Add.
# ============================================================
import os
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import sys
import shutil
import subprocess
from pathlib import Path

# 1. Clone repo dari GitHub (atau pull jika sudah ada di sesi ini)
REPO_URL = "https://github.com/faizhuda/pcd-kelompok-17.git"
PROJECT_DIR = Path("/kaggle/working/pcd-kelompok-17")
if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=False)

# 2. Working directory ke root project + tambah ke sys.path
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# 3. Dependency inti SUDAH pre-installed di Kaggle -> tidak ada pip install.

# 4. Dataset gambar (read-only, hasil + Add Data)
# Auto-detect: Kaggle bisa mount di /kaggle/input/<slug> atau
# /kaggle/input/datasets/<user>/<slug> tergantung cara attach.
_DATASET_SLUG = 'fruit-and-vegetable-disease-healthy-vs-rotten'
_candidates = [
    Path('/kaggle/input') / _DATASET_SLUG,
    Path('/kaggle/input/datasets/muhammad0subhan') / _DATASET_SLUG,
]
RAW_DATA_DIR = next((p for p in _candidates if p.exists()), None)
if RAW_DATA_DIR is None:
    # Fallback: cari folder mana saja di /kaggle/input yang berisi gambar dataset
    for _p in Path('/kaggle/input').rglob(_DATASET_SLUG):
        if _p.is_dir():
            RAW_DATA_DIR = _p
            break
assert RAW_DATA_DIR is not None, "Dataset belum di-attach. + Add Data > cari dataset > Add."

# 5. Auto-restore hasil notebook sebelumnya (untuk notebook 03 & 04).
#    Attach output run lama via: + Add Data > Your Work / Dataset bersama.
def restore_previous_outputs():
    # Kaggle mounts notebook outputs di /kaggle/input/notebooks/<user>/<notebook>/
    # sehingga perlu rglob, bukan glob satu level.
    restored = []
    for repo in Path("/kaggle/input").rglob("pcd-kelompok-17"):
        if not repo.is_dir():
            continue
        for sub in ("results", "data/processed"):
            src_dir = repo / sub
            if src_dir.exists():
                shutil.copytree(src_dir, PROJECT_DIR / sub, dirs_exist_ok=True)
                restored.append(str(src_dir))
    return restored

restored = restore_previous_outputs()
print("Project :", PROJECT_DIR)
print("Dataset :", RAW_DATA_DIR)
print("Restore :", restored or "(mulai dari nol)")


Cloning into '/kaggle/working/pcd-kelompok-17'...


Project : /kaggle/working/pcd-kelompok-17
Dataset : /kaggle/input/datasets/muhammad0subhan/fruit-and-vegetable-disease-healthy-vs-rotten
Restore : (mulai dari nol)


In [2]:
import os
import sys
from pathlib import Path

# Setup cell sudah chdir ke PROJECT_DIR & menambah sys.path (Kaggle-only).
ROOT = Path("/kaggle/working/pcd-kelompok-17")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.experiments import (
    run_classical_scenario,
    run_mcnemar_pair,
    select_best_enhancement,
)
from src.utils import build_dataset_index, get_project_paths, make_splits, set_seed

set_seed(42)
paths = get_project_paths()
# Split di-regenerate dari dataset (deterministik, SEED=42) - tidak baca splits.json
# RAW_DATA_DIR sudah di-set setup cell (auto-detect path Kaggle)
train, val, test = make_splits(build_dataset_index(RAW_DATA_DIR))

# Stratified sub-sampling untuk memotong running time klasifikasi klasik (S1-S9)
# dari ~5 jam menjadi ~1 jam. Pola perbandingan relatif antar-skenario tetap konsisten.
def sub_sample(df, max_per_group):
    return (
        df.groupby(['commodity', 'label'], group_keys=False)
        .apply(lambda g: g.sample(min(len(g), max_per_group), random_state=42))
        .reset_index(drop=True)
    )

train = sub_sample(train, 150)
val = sub_sample(val, 50)
test = sub_sample(test, 50)
cache_dir = paths["data_processed"]
metrics_dir = paths["metrics"]
figures_dir = paths["figures_confusion"]
models_dir = paths["models"]
print('Sub-sampled counts:', len(train), len(val), len(test))


E0000 00:00:1781061133.745399      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781061133.825476      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781061134.459862      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781061134.459922      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781061134.459925      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781061134.459928      16 computation_placer.cc:177] computation placer already registered. Please check linka

Sub-sampled counts: 4119 1240 1241


## Skenario 1-4: Baseline, Restorasi (SSR), Enhancement

- **S1** = baseline mentah (tanpa restorasi, tanpa enhancement)
- **S2** = + restorasi SSR (isolasi efek koreksi pencahayaan vs S1)
- **S3/S4** = SSR + CLAHE / gamma. E* dipilih dari S2-S4 (val F1).

In [3]:
val_f1_map = {}
scenario_results = {}

for sid in range(1, 5):
    print(f"\n=== Skenario {sid} ===")
    res = run_classical_scenario(
        sid, train, val, test,
        metrics_dir, figures_dir, models_dir, cache_dir,
    )
    scenario_results[sid] = res
    # E* dipilih di antara skenario ber-SSR (S2 none, S3 clahe, S4 gamma).
    # S1 = baseline mentah (tanpa restorasi) -> tidak ikut pemilihan enhancement.
    if sid >= 2:
        val_f1_map[res["enhancement"]] = res["val_f1"]
    print(f"Val F1: {res['val_f1']:.4f} | Test F1: {res['test_metrics']['f1_weighted']:.4f}")

best_enh = select_best_enhancement(val_f1_map, metrics_dir)
print(f"\nE* (enhancement terbaik): {best_enh}")



=== Skenario 1 ===


Extracting features: 100%|██████████| 1241/1241 [01:42<00:00, 12.14it/s]


Val F1: 0.8992 | Test F1: 0.8985

=== Skenario 2 ===


Extracting features: 100%|██████████| 1241/1241 [01:42<00:00, 12.16it/s]


Val F1: 0.8548 | Test F1: 0.8759

=== Skenario 3 ===


Extracting features: 100%|██████████| 1241/1241 [01:46<00:00, 11.63it/s]


Val F1: 0.8677 | Test F1: 0.8735

=== Skenario 4 ===


Extracting features: 100%|██████████| 1241/1241 [01:45<00:00, 11.75it/s]


Val F1: 0.8702 | Test F1: 0.8775

E* (enhancement terbaik): gamma


## Skenario 5-9: Segmentasi, Ablasi Fitur, Random Forest

- **S5** = E* + segmentasi, semua fitur, SVM (pipeline klasik penuh)
- **S6/S7/S8** = ablasi fitur (warna saja / tekstur saja / bentuk saja)
- **S9** = S5 dengan Random Forest (perbandingan classifier + feature importance)

In [4]:
for sid in range(5, 10):
    print(f"\n=== Skenario {sid} ===")
    res = run_classical_scenario(
        sid, train, val, test,
        metrics_dir, figures_dir, models_dir, cache_dir,
    )
    scenario_results[sid] = res
    print(f"Test F1: {res['test_metrics']['f1_weighted']:.4f}")



=== Skenario 5 ===


Extracting features: 100%|██████████| 1241/1241 [02:06<00:00,  9.80it/s]


Test F1: 0.8864

=== Skenario 6 ===
Test F1: 0.8313

=== Skenario 7 ===
Test F1: 0.7881

=== Skenario 8 ===
Test F1: 0.6033

=== Skenario 9 ===
Test F1: 0.8767


## Uji Signifikansi McNemar (isolasi tiap tahap)

In [5]:
from src.utils import read_best_enhancement

best_enh = read_best_enhancement(metrics_dir)
# Skenario no-seg yang memakai E* (untuk isolasi efek segmentasi):
# none -> S2, clahe -> S3, gamma -> S4.
enh_noseg_sid = {"none": 2, "clahe": 3, "gamma": 4}[best_enh]
y_true = scenario_results[1]["y_test"]

# 1. Efek restorasi SSR: S2 (ssr) vs S1 (mentah)
run_mcnemar_pair(
    "S2 vs S1 (SSR)", "S2", "S1",
    y_true, scenario_results[2]["y_pred"], scenario_results[1]["y_pred"], metrics_dir,
)

# 2. Efek enhancement E*: S{E*} vs S2 (hanya bermakna bila E* != none)
if enh_noseg_sid != 2:
    run_mcnemar_pair(
        f"E*({best_enh}) vs S2", f"S{enh_noseg_sid}", "S2",
        y_true, scenario_results[enh_noseg_sid]["y_pred"], scenario_results[2]["y_pred"], metrics_dir,
    )

# 3. Efek segmentasi: S5 (E*+seg) vs E* tanpa seg
run_mcnemar_pair(
    "S5 vs E*-noseg (segmentasi)", "S5", f"S{enh_noseg_sid}",
    y_true, scenario_results[5]["y_pred"], scenario_results[enh_noseg_sid]["y_pred"], metrics_dir,
)
print("McNemar CNN (S10/S11 vs S5) dijalankan di notebook 03.")


McNemar CNN (S10/S11 vs S5) dijalankan di notebook 03.


In [6]:
import pandas as pd
sig_path = metrics_dir / "significance_tests.csv"
if sig_path.exists():
    display(pd.read_csv(sig_path))


,comparison,model_a,model_b,statistic,p_value,conclusion
0,S10 vs S5 (CNN vs SVM),S10,S5,324.375433,1.613828e-72,signifikan
1,S11 vs S1 (CNN-raw vs SVM-raw),S11,S1,151.803509,6.994728e-35,signifikan
2,S10 vs S11 (full vs raw CNN),S10,S11,36.900901,1.242885e-09,signifikan
3,S2 vs S1 (SSR),S2,S1,4.142045,4.183059e-02,signifikan
4,E*(gamma) vs S2,S4,S2,0.011628,9.141283e-01,tidak signifikan
5,S5 vs E*-noseg (segmentasi),S5,S4,1.030928,3.099408e-01,tidak signifikan


## Re-evaluasi pada Full Test Set (n=4391)

**Perbaikan metodologis:** S1–S9 di atas dievaluasi pada test sub-sampled (n≈1241, 50 per grup) sementara S10/S11 (CNN) menggunakan full test set (n=4391). Sel ini menjalankan ulang prediksi semua S1–S9 pada full test sehingga perbandingan F1 antar-model menjadi *apple-to-apple*.

- **Train/val tetap sub-sampled** — model tidak dilatih ulang dari nol.
- **Hanya test yang diganti** ke full split deterministik (SEED=42).
- `scenario_XX.csv` dan `predictions_s5.csv` diperbarui dengan hasil baru.
- Uji McNemar S2-vs-S1/E*-vs-S2/S5-vs-E* di atas tetap valid (menguji arah perbedaan relatif, bukan angka absolut); McNemar SVM-vs-CNN sudah memakai full test sejak nb03.

In [7]:
import time

# Full test split (no sub-sampling) - train/val remain sub-sampled
_, _, test_full = make_splits(build_dataset_index(RAW_DATA_DIR))
print(f'Full test: n={len(test_full)}')

set_seed(42)
summary_reeval = []

for sid in range(1, 10):
    print(f'\n=== Re-evaluasi Skenario {sid} (full test) ===')
    t0 = time.perf_counter()
    res_full = run_classical_scenario(
        sid, train, val, test_full,
        metrics_dir, figures_dir, models_dir, cache_dir,
    )
    elapsed = time.perf_counter() - t0
    f1_full = res_full['test_metrics']['f1_weighted']
    f1_sub  = scenario_results[sid]['test_metrics']['f1_weighted']
    n_full  = len(res_full['y_test'])
    summary_reeval.append({
        'Skenario': f'S{sid}',
        'F1 sub-sampled': round(f1_sub, 4),
        'F1 full test': round(f1_full, 4),
        'n_test': n_full,
    })
    print(f'F1: {f1_sub:.4f} → {f1_full:.4f} | n={n_full} | {elapsed:.1f}s')

import pandas as pd
print('\nRingkasan perubahan F1 setelah re-evaluasi full test:')
display(pd.DataFrame(summary_reeval))
print('\nSemua scenario_XX.csv dan predictions_s5.csv diperbarui.')


Full test: n=4391

=== Re-evaluasi Skenario 1 (full test) ===


Extracting features: 100%|██████████| 4391/4391 [05:38<00:00, 12.98it/s]


F1: 0.8985 → 0.8998 | n=4391 | 354.3s

=== Re-evaluasi Skenario 2 (full test) ===


Extracting features: 100%|██████████| 4391/4391 [05:58<00:00, 12.25it/s]


F1: 0.8759 → 0.8593 | n=4391 | 374.8s

=== Re-evaluasi Skenario 3 (full test) ===


...

F1: 0.8775 → 0.8633 | n=4391 | 384.3s

=== Re-evaluasi Skenario 5 (full test) ===


Extracting features: 100%|██████████| 4391/4391 [07:24<00:00,  9.88it/s]


F1: 0.8864 → 0.8735 | n=4391 | 461.7s

=== Re-evaluasi Skenario 6 (full test) ===
F1: 0.8313 → 0.8328 | n=4391 | 13.0s

=== Re-evaluasi Skenario 7 (full test) ===
F1: 0.7881 → 0.7715 | n=4391 | 5.4s

=== Re-evaluasi Skenario 8 (full test) ===
F1: 0.6033 → 0.5723 | n=4391 | 4.5s

=== Re-evaluasi Skenario 9 (full test) ===
F1: 0.8767 → 0.8720 | n=4391 | 2.9s

Ringkasan perubahan F1 setelah re-evaluasi full test:


,Skenario,F1 sub-sampled,F1 full test,n_test
0,S1,0.8985,0.8998,4391
1,S2,0.8759,0.8593,4391
2,S3,0.8735,0.8589,4391
3,S4,0.8775,0.8633,4391
4,S5,0.8864,0.8735,4391
5,S6,0.8313,0.8328,4391
6,S7,0.7881,0.7715,4391
7,S8,0.6033,0.5723,4391
8,S9,0.8767,0.8720,4391



Semua scenario_XX.csv dan predictions_s5.csv diperbarui.
